# 02 Rebuild dataset from raw BDD100K zip files

This notebook rebuilds the dataset **locally in `/content`** from the raw BDD100K archives in `EcoCAR/downloads`, then saves **only** the packaged tarball and `paths_config.yaml` to Drive for `notebook00`.

`notebook00` does **not** require external segmentation-mask files for training. It trains on:
- YOLO-format vehicle detection labels
- geometric lane targets parsed from raw BDD100K JSON (`poly2d` / fallback `seg2d`)

`seg_maps_root` is still recorded in `paths_config.yaml` for optional future use, but it is not required by the current `notebook00` training pipeline.


In [2]:
from pathlib import Path
import json, os, sys, shutil, tarfile

from google.colab import drive
drive.mount('/content/drive')

ECOCAR_ROOT = Path('/content/drive/MyDrive/EcoCAR')
PROJECT_ROOT = ECOCAR_ROOT / 'DETR_GeoLane_pipeline'
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path('/content/DETR_GeoLane_pipeline')
assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DOWNLOADS = ECOCAR_ROOT / 'downloads'
RAW_ROOT = Path('/content/bdd100k_raw')
OUTPUT_ROOT = Path('/content/bdd100k_vehicle5_build')
PACKAGE_TAR = ECOCAR_ROOT / 'datasets' / 'bdd100k_vehicle5.tar'
GLOBAL_PATHS_CONFIG = ECOCAR_ROOT / 'paths_config.yaml'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DOWNLOADS    =', DOWNLOADS)
print('RAW_ROOT     =', RAW_ROOT)
print('LOCAL BUILD  =', OUTPUT_ROOT)
print('PACKAGE_TAR  =', PACKAGE_TAR)
print('GLOBAL_PATHS_CONFIG =', GLOBAL_PATHS_CONFIG)


Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/EcoCAR/DETR_GeoLane_pipeline
DOWNLOADS    = /content/drive/MyDrive/EcoCAR/downloads
RAW_ROOT     = /content/bdd100k_raw
LOCAL BUILD  = /content/bdd100k_vehicle5_build
PACKAGE_TAR  = /content/drive/MyDrive/EcoCAR/datasets/bdd100k_vehicle5.tar
GLOBAL_PATHS_CONFIG = /content/drive/MyDrive/EcoCAR/paths_config.yaml


In [3]:
from src.data_prep import inspect_download_archives

archive_preview = inspect_download_archives(DOWNLOADS)
for name, members in archive_preview.items():
    print('\n' + '='*90)
    print(name)
    if not members:
        print('NOT FOUND')
    else:
        for m in members[:30]:
            print(m)


bdd100k_labels.zip
100k/
100k/train/
100k/train/6866acb3-cf17e759.json
100k/train/1a4e992d-79679eee.json
100k/train/3f5f0305-e44b08eb.json
100k/train/a37ed628-303a9552.json
100k/train/50d73ea5-aeda4064.json
100k/train/902c76f4-39ebe859.json
100k/train/31189da5-f69b5427.json
100k/train/2c272437-250ddee3.json
100k/train/622af22a-a98ff126.json
100k/train/059249ba-f90bb77e.json
100k/train/0cd9079a-ca3c4af8.json
100k/train/3f29ef75-bee2fba1.json
100k/train/34a50e9e-82bfcac8.json
100k/train/538ec15f-1c4537de.json
100k/train/15213755-77609c0e.json
100k/train/33b7f558-21c7a84f.json
100k/train/17587a71-e84fb69c.json
100k/train/61ba46bb-f9a1d5df.json
100k/train/10c8f3c5-d49ebc5f.json
100k/train/41456314-522a8f9d.json
100k/train/3670e3ff-15327d62.json
100k/train/a0905818-f8b4b33f.json
100k/train/954b31d2-3cc3f1fa.json
100k/train/5e7717df-ff7a5378.json
100k/train/5b27653c-2367eb0b.json
100k/train/33fe529e-191a2404.json
100k/train/9a7b697a-8881200b.json
100k/train/67c665d7-5d43cd3d.json

bdd100k_i

In [4]:
from src.data_prep import rebuild_dualpath_dataset

summary = rebuild_dualpath_dataset(
    downloads_dir=DOWNLOADS,
    raw_root=RAW_ROOT,
    output_root=OUTPUT_ROOT,
    force_reextract=False,
)

print(json.dumps(summary, indent=2))

{
  "train_json": "/content/bdd100k_raw/100k/train",
  "val_json": "/content/bdd100k_raw/100k/val",
  "train_image_dir": "/content/bdd100k_raw/100k/train",
  "val_image_dir": "/content/bdd100k_raw/100k/val",
  "lane_json": "/content/bdd100k_raw/100k/train",
  "seg_maps_root": "/content/bdd100k_raw/color_labels/train",
  "train_image_count": 70000,
  "val_image_count": 10000,
  "train_counts": {
    "files_written": 70000,
    "skipped_no_box": 219,
    "car": 713917,
    "truck": 30003,
    "bus": 11684,
    "motorcycle": 3002,
    "bicycle": 7225
  },
  "val_counts": {
    "files_written": 10000,
    "skipped_no_box": 36,
    "car": 102506,
    "truck": 4245,
    "bus": 1597,
    "motorcycle": 452,
    "bicycle": 1007
  },
  "yaml_path": "/content/bdd100k_vehicle5_build/bdd100k_vehicle5.yaml",
  "paths_config": "/content/bdd100k_vehicle5_build/paths_config.yaml"
}


In [5]:
# Package notebook02 output for notebook00
from pathlib import Path
import shutil, tarfile, os

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PACKAGE_TAR.parent.mkdir(parents=True, exist_ok=True)

# Copy dataset-local paths_config.yaml to the global location used by src.config
local_paths_cfg = OUTPUT_ROOT / 'paths_config.yaml'
if local_paths_cfg.exists():
    shutil.copy2(local_paths_cfg, GLOBAL_PATHS_CONFIG)
    print('Copied paths_config to', GLOBAL_PATHS_CONFIG)
else:
    print('WARNING: paths_config.yaml missing at', local_paths_cfg)

# notebook00 should consume only the packaged tar + global paths_config from Drive.
# Build happens in /content; Drive stores the final tar artifact only.
with tarfile.open(PACKAGE_TAR, 'w', dereference=True) as tar:
    for item in sorted(OUTPUT_ROOT.iterdir()):
        tar.add(item, arcname=item.name)

print('Packaged dataset tar:', PACKAGE_TAR)
print('Tar size (GB):', round(PACKAGE_TAR.stat().st_size / 1024**3, 3))
print('LOCAL BUILD contents:', sorted([p.name for p in OUTPUT_ROOT.iterdir()]))
print('Drive receives only:')
print('  -', PACKAGE_TAR)
print('  -', GLOBAL_PATHS_CONFIG)


Copied paths_config to /content/drive/MyDrive/EcoCAR/paths_config.yaml
Packaged dataset tar: /content/drive/MyDrive/EcoCAR/datasets/bdd100k_vehicle5.tar
Tar size (GB): 4.618
LOCAL BUILD contents: ['bdd100k_vehicle5.yaml', 'images', 'labels', 'paths_config.yaml']
Drive receives only:
  - /content/drive/MyDrive/EcoCAR/datasets/bdd100k_vehicle5.tar
  - /content/drive/MyDrive/EcoCAR/paths_config.yaml


In [6]:
from collections import Counter

def scan_raw_ids(label_dir, max_files=2000):
    counter = Counter()
    txts = sorted(label_dir.glob('*.txt'))[:max_files]
    for p in txts:
        for line in p.read_text().splitlines():
            s = line.strip().split()
            if len(s) >= 5:
                counter[int(float(s[0]))] += 1
    return counter

train_counter = scan_raw_ids(OUTPUT_ROOT / 'labels' / 'train', 3000)
val_counter = scan_raw_ids(OUTPUT_ROOT / 'labels' / 'val', 1000)
print('train raw ids ->', dict(sorted(train_counter.items())))
print('val raw ids   ->', dict(sorted(val_counter.items())))
print('expected ids  -> {0: car, 1: truck, 2: bus, 3: motorcycle, 4: bicycle}')

train raw ids -> {0: 30566, 1: 1343, 2: 503, 3: 120, 4: 322}
val raw ids   -> {0: 9840, 1: 495, 2: 172, 3: 54, 4: 87}
expected ids  -> {0: car, 1: truck, 2: bus, 3: motorcycle, 4: bicycle}


In [7]:
yaml_path = OUTPUT_ROOT / 'bdd100k_vehicle5.yaml'
paths_cfg = OUTPUT_ROOT / 'paths_config.yaml'
print('Dataset YAML:', yaml_path)
print(yaml_path.read_text())
print('Paths config:', paths_cfg)
print(paths_cfg.read_text())

Dataset YAML: /content/bdd100k_vehicle5_build/bdd100k_vehicle5.yaml
path: /content/bdd100k_vehicle5_build
train: images/train
val: images/val
nc: 5
names:
  0: car
  1: truck
  2: bus
  3: motorcycle
  4: bicycle

Paths config: /content/bdd100k_vehicle5_build/paths_config.yaml
dataset_root: /content/bdd100k_vehicle5_build
raw_root: /content/bdd100k_raw
lane_json: /content/bdd100k_raw/100k/train
seg_maps_root: /content/bdd100k_raw/color_labels/train

